#### Aggregations in Apache Spark DataFrame/Dataset

![image_1775378873059.png](./image_1775378873059.png "image_1775378873059.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, sum, avg, min, max, countDistinct, collect_list, collect_set


data = [
    ("Alice",   "IN", "Electronics", 5000),
    ("Bob",     "IN", "Clothing",    3000),
    ("Charlie", "IN", "Electronics", 7000),
    ("Diana",   "US", "Clothing",    4000),
    ("Eve",     "US", "Electronics", 8000),
    ("Frank",   "US", "Clothing",    6000),
    ("Grace",   "EU", "Electronics", 9000),
    ("Hank",    "EU", "Clothing",    2000),
    ("Ivy",     "EU", "Electronics", 4500),
    ("Jack",    "IN", "Electronics", 7000),  # duplicate sales value
]

df = spark.createDataFrame(data, ["name", "region", "category", "sales"])

#### 1. Basic Aggregation Functions

In [0]:
from pyspark.sql.functions import count, sum, avg, min, max, countDistinct

df.groupBy("region").count()
df.groupBy("region").sum("sales")
df.groupBy("region").avg("sales")
df.groupBy("region").min("sales")
df.groupBy("region").max("sales")

# Multiple aggregations at once
result = df.groupBy("region").agg(
    count("*").alias("total_rows"),
    sum("sales").alias("total_sales"),
    avg("sales").alias("avg_sales"),
    countDistinct("name").alias("unique_customers")
)

In [0]:
result.explain()

#### 2. Global Aggregations (entire DataFrame)

In [0]:
df.agg(
    count("*").alias("total_rows"),
    count("sales").alias("non_null_sales"),
    countDistinct("region").alias("unique_regions"),
    sum("sales").alias("total_sales"),
    avg("sales").alias("avg_sales"),
    min("sales").alias("min_sales"),
    max("sales").alias("max_sales")
).show()

#### 3. GroupBy Aggregations

In [0]:
df.groupBy("region").agg(
    count("*").alias("total_rows"),
    count("sales").alias("non_null_sales"),
    sum("sales").alias("total_sales"),
    avg("sales").alias("avg_sales"),
    min("sales").alias("min_sales"),
    max("sales").alias("max_sales")
).show()

##### Multiple columns groupBy

In [0]:
df.groupBy("region", "category").agg(
    count("*").alias("No.of Sales"),
    sum("sales").alias("Total Sales"),
    avg("sales").alias("Avg. Sales"),
).orderBy("region", "category").show()

#### 4. collect_list vs collect_set

In [0]:
df.groupBy("region").agg(
    collect_list("sales").alias("Sales By Category"),
    collect_set("sales").alias("Unique Sales By Category")
).show()

#### 5. Aggregation with Filtering — filter inside agg

In [0]:
from pyspark.sql.functions import when, col

df.groupBy("region").agg(
    count("*").alias("Total Count"),
    sum(when(col("sales") > 5000, 1).otherwise(0)).alias("High Sales Count"),
    sum(when(col("category") == "Electronics", col("sales"))).alias("Electronic Sales")
).show()

In [0]:
from pyspark.sql.functions import when, col

# Count only rows where sales > 5000
df.groupBy("region").agg(
    count("*").alias("total_count"),
    sum(when(col("sales") > 5000, 1).otherwise(0)).alias("high_sales_count"),
    sum(when(col("category") == "Electronics", col("sales"))).alias("electronics_sales")
).show()

#### 6. Pivot — Rows to Columns

In [0]:
df.groupBy("category").pivot("region").agg(sum("sales")).show()

> Always specify pivot values explicitly — without them Spark runs an extra job to discover distinct values.


#### 7. Rollup & Cube — Hierarchical Aggregations

##### rollup — subtotals along hierarchy (left to right)

In [0]:
df.rollup("region", "category").agg(
  sum("sales").alias("total_sales")
).show()

##### cube — subtotals for ALL combinations

In [0]:
df.cube("region", "category").agg(
    sum("sales").alias("Total Sales")
).orderBy("region", "category").show()

![image_1775377796143.png](./image_1775377796143.png "image_1775377796143.png")

#### 8. approx_count_distinct — Performance Optimization


In [0]:
from pyspark.sql.functions import approx_count_distinct

# countDistinct is exact but expensive (shuffle)
df.agg(countDistinct("name").alias("exact_count")).show()

# approx_count_distinct uses HyperLogLog — much faster on large data
df.agg(approx_count_distinct("name", rsd=0.05).alias("approx_count")).show()

##### countDistinct with Multiple Columns

Single Column vs Multiple Columns


In [0]:
from pyspark.sql.functions import countDistinct

data = [
    ("Alice", "IN", "Electronics"),
    ("Alice", "IN", "Clothing"),     # same name+region, different category
    ("Alice", "US", "Electronics"),  # same name, different region
    ("Bob",   "IN", "Electronics"),
    ("Bob",   "IN", "Electronics"),  # exact duplicate
    ("Bob",   "EU", "Clothing"),
]
df1 = spark.createDataFrame(data, ["name", "region", "category"])

In [0]:
df1.agg(
    countDistinct("name").alias("distinct_names"),                      # unique names
    countDistinct("region").alias("distinct_regions"),                  # unique regions
    countDistinct("name", "region").alias("distinct_name_region"),      # unique name+region combos
    countDistinct("name", "region", "category").alias("distinct_all")   # unique all 3 combos
).show()

##### With groupBy + countDistinct

In [0]:
# How many distinct categories does each region have?
df1.groupBy("region").agg(
    countDistinct("category").alias("distinct_categories"),
    countDistinct("name", "category").alias("distinct_name_category_combos")
).show()

> countDistinct vs count + distinct
Both produce the same result but countDistinct is more concise:

In [0]:
# These are equivalent
df1.agg(countDistinct("name", "region")).show()

df1.select("name", "region").distinct().count()  # same result, two operations